# The Measurement Ceiling

How much of the 2024 answer key is signal rather than noise, and what fraction of that
signal each rung of the ladder actually reaches. Phase M's job is to make the sentence
"the model reaches X% of the ceiling" mean something specific — X% of *which* ceiling,
measured on *which* hitters, with *what* left over.

**Everything below is post-selection descriptive.** The pre-registered platoon gate ran on
2024 and failed (`docs/research-manifest.md`, 2026-08-20). Every number here describes a
measurement problem that was discovered afterward; none of it is evidence for or against
the pre-registered claim, and none of it may be read as a test.

**2025 is sealed.** No cell here reads, scores, or refits on it.

This notebook **reads committed artifacts and recomputes nothing**. Everything is produced
by `src/analysis/measurement_ceiling_report.py` and `src/analysis/measurement_ceiling.py`, gated by
`tests/test_measurement_ceiling.py` (the planted-recovery self-checks) and `tests/test_measurement_ceiling_report.py`
(the reproduction gates). Re-run with:

```
PYTHONPATH=. python -m src.analysis.measurement_ceiling_report
```

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.analysis import claim1_eval

## Configuration

Artifact locations and the orderings used below. The scored arm is the `embedding_sgd_sgd_lr1`
five-seed ensemble (canonical build since 2026-09-03) and the season is 2024 throughout.

In [2]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 140)

PHASE_M_DIR = REPO_ROOT / "results/measurement_ceiling"
PHASE_E_DIR = REPO_ROOT / "results/model_evaluation"
PHASE_O_DIR = REPO_ROOT / "results/hyperparameter_tuning"

EVAL_SEASON = 2024
ROUTE_ORDER = ["B_prime", "A", "B"]
STRATUM_ORDER = list(claim1_eval.STRATUM_NAMES)


def load_m(name):
    """Read one committed Phase M table by filename."""
    return pd.read_csv(PHASE_M_DIR / name)


def load_json_m(name):
    """Read one committed Phase M JSON artifact by filename."""
    return json.loads((PHASE_M_DIR / name).read_text())


summary = load_json_m("measurement_ceiling_summary.json")
routes_json = load_json_m("routes.json")

## Which fallback rules fired

The spec pre-registered a fallback for every contingency it could name (§10). Any rule that
fired is flagged here, at the top, before a single number is read — a fired rule changes
what the numbers below mean.

In [3]:
for rule in summary["fallback_rules_fired"]:
    print("FIRED:", rule, end="\n\n")
print("route rule (frozen before any Phase M number was read):")
print(json.dumps(summary["route_rule"], indent=2))

route rule (frozen before any Phase M number was read):
{
  "primary": "B_prime",
  "always_reported": [
    "A"
  ],
  "provenance_only": [
    "B"
  ],
  "never_primary": [
    "C"
  ],
  "rationale": "Route A's fragility is structural \u2014 a near-total cancellation where a 3% noise-model error swings tau2 by ~100% \u2014 and that was known before any number was read.",
  "frozen": "docs/phase-m-spec.md \u00a7M.0, decision log 2026-08-30"
}


## M.6 — which hitters are actually being compared

E.5 (the platoon differential) and F.5 (the pooled level score) scored different
populations, and every "fraction of the ceiling" statement silently assumed they were the
same one. They are not — but E.5 turns out to be a strict **subset** of F.5, so the
intersection *is* E.5's population and M.0–M.2 need no re-restriction. F.5's 72 extra
hitters are hitters with a pooled score but no usable two-sided differential.

F.5's population is **rebuilt** through `pooled`'s own path rather than transcribed, so
its coverage counts are a reproduction check, not a copy.

In [4]:
population = load_m("population.csv")
print(json.dumps({key: value for key, value in summary["population"].items()
                  if key != "pooled_rebuilt_coverage"}, indent=2))
population.head()

{
  "n_platoon": 545,
  "n_pooled": 617,
  "n_intersection": 545,
  "n_platoon_only": 0,
  "n_pooled_only": 72,
  "platoon_is_subset_of_pooled": true
}


,batter,in_platoon,in_pooled,in_intersection,denom_L,denom_R,stand,stratum,pooled_denom,post_selection_descriptive
0,444482,True,True,True,26.0,234.0,L,high,260,True
1,453568,True,True,True,114.0,383.0,L,high,497,True
2,455117,True,True,True,43.0,101.0,R,high,144,True
3,456781,True,True,True,118.0,191.0,R,high,309,True
4,457705,True,True,True,141.0,373.0,R,high,514,True


## M.0 — the ceiling, by route

Reliability is true-talent variance over observed variance; its square root is the highest
rank correlation any predictor could reach against this answer key. The routes differ only
in how τ² (the true within-stand differential variance) is estimated:

| Route | τ² from | Role |
|---|---|---|
| **B′** | the C.2 bivariate fit, refit on the M.6 intersection | **primary** (pre-registered) |
| A | 2024 empirical subtraction: observed within-stand variance − mean sampling variance | sensitivity, always reported with its fragility band |
| B | the committed, unrestricted C.2 fit | provenance only; the B→B′ delta is the population diagnostic |
| C | split-half + Spearman-Brown | never an estimator — a bounded diagnostic, below |

The rule was frozen before any of these numbers was read. Route A is not primary because
its fragility is *structural*: it is a near-total cancellation of two numbers that agree to
three decimal places, so a 3% error in the noise model roughly doubles τ².

In [5]:
routes = pd.DataFrame([dict(route=name, **routes_json["routes"][name])
                       for name in ROUTE_ORDER])
routes[["route", "tau2", "mean_sampling_var", "reliability", "ceiling_rank_corr",
        "achieved_rank_corr", "fraction_of_ceiling", "degenerate"]].round(6)

,route,tau2,mean_sampling_var,reliability,ceiling_rank_corr,achieved_rank_corr,fraction_of_ceiling,degenerate
0,B_prime,0.000393,0.004078,0.087884,0.296452,0.162633,0.548600,False
1,A,0.000117,0.004078,0.027922,0.167100,0.162633,0.973270,False
2,B,0.000590,0.004078,0.126442,0.355587,0.162633,0.457366,False


### The B→B′ population diagnostic

B and B′ are the same estimator on the same three-season window; they differ **only** in
which hitters the past seasons are restricted to. So the gap between them is a measurement
of how much of the A-vs-B disagreement is a population difference rather than an estimator
difference.

Note what B′ conditions on: restricting past seasons to hitters who *reached* the 2024 eval
population conditions on survival to 2024. That is the correct population for a claim about
2024 eval hitters, and it is **not** a general-population τ².

In [6]:
print(json.dumps(routes_json["b_to_b_prime_diagnostic"], indent=2))

{
  "tau2_B": 0.0005903354341684647,
  "tau2_B_prime": 0.0003929668596653433,
  "relative_drop": 0.33433292850043983,
  "share_of_B_to_A_gap_closed": 0.41710792949507175,
  "reading": "a large drop toward A means selection into the 2024 eval population explains the gap; no material drop means the remaining A/B' gap is window or estimator",
  "verdict": "selection explains part of the gap; the remainder is window or estimator",
  "conditioning_label": "Route B' restricts PAST seasons to hitters who reached the 2024 eval population, which conditions on survival to 2024. That is the correct population for a claim about 2024 eval hitters, and it is not a general-population tau2."
}


### Route A's fragility band, and a caveat on the ceiling formula itself

The band recomputes Route A with the sampling-variance term scaled ×0.97 and ×1.03. It is
reported *with* Route A everywhere, never separately.

The band does not merely widen the estimate — **it breaks it**. At ×0.97 the ceiling rises
from 0.167 to 0.239, most of the distance to B′. At ×1.03 τ² goes *negative* and the route
returns no ceiling at all: a 3% error in the noise model, in the unfavourable direction,
leaves Route A saying there is no measurable true differential. Nothing is clipped to zero
to hide this; a non-positive τ² is emitted negative and flagged (spec §9.2). This is why the
route rule made B′ primary before any of these numbers were read.

**Methods note — the analytic ceiling is not conservative here.** `sqrt(reliability)` is
derived under joint normality and bounds the *Pearson* correlation; the project scores with
a *rank* correlation. Textbook reasoning says the rank ceiling should sit slightly below
the analytic one, and under a homoscedastic simulation it does (−3.6%). Under the real 2024
exposure profile it does not: the sampling-variance skew across hitters is ~36×, rank
correlation is robust to the heavy-tailed observations low-exposure hitters contribute, and
that robustness buys back more than joint normality costs. The measured rank ceiling comes
in **above** the analytic one.

Consequence, stated plainly: **every fraction-of-ceiling figure in this notebook runs a few
percent relatively HIGH.** The Monte-Carlo rank ceiling is reported beside the analytic one
so the size of the discrepancy is visible rather than assumed away. The Pearson check
matches the analytic value exactly, so the reliability→ceiling map itself is verified; it
is the rank-vs-Pearson step that is approximate.

In [7]:
band = routes_json["fragility_band"]
print("Route A fragility band (sampling variance x0.97 / x1.03)")
print(pd.DataFrame({"sampling_scale": band["scales"], "tau2": band["tau2"],
                    "ceiling_rank_corr": band["ceiling_rank_corr"]}).round(6)
      .to_string(index=False))
print("\nceiling range over the non-degenerate scales:",
      [round(value, 4) for value in band["ceiling_range_finite_only"]])
print("a scale in the band drove tau2 non-positive:", band["any_degenerate"])

monte_carlo = routes_json["monte_carlo_rank_ceiling"]
print("\nanalytic vs measured rank ceiling, at the primary route's tau2:")
print(pd.Series({
    "analytic sqrt(reliability)": monte_carlo["ceiling_rank_corr"],
    "simulated Pearson (mean)": monte_carlo["mc_pearson_mean"],
    "simulated Spearman (mean)": monte_carlo["mc_spearman_mean"],
    "simulated Spearman (sd)": monte_carlo["mc_spearman_sd"],
}).round(4).to_string())

Route A fragility band (sampling variance x0.97 / x1.03)
 sampling_scale      tau2  ceiling_rank_corr
           1.00  0.000117           0.167100
           0.97  0.000240           0.238924
           1.03 -0.000005                NaN

ceiling range over the non-degenerate scales: [0.1671, 0.2389]
a scale in the band drove tau2 non-positive: True

analytic vs measured rank ceiling, at the primary route's tau2:
analytic sqrt(reliability)    0.2965
simulated Pearson (mean)      0.2978
simulated Spearman (mean)     0.3093
simulated Spearman (sd)       0.0420


### What the ceiling means in plate appearances

The same τ² re-expressed as PA*, the per-side exposure at which reliability reaches 0.5 —
the point where half of what you observe is signal.

Two conventions are reported because they differ by roughly 2× and a reader will assume the
second. `pa_star_weak_side` charges only the PAs faced vs LHP, which is C.2's own
`implied_split_constant` convention and the one the handoff's "~430" and ">2000" figures
were on; both are confirmed here. `pa_star_both_sides` charges both sides growing together,
which is how exposure actually accumulates.

In [8]:
stabilization = routes_json["stabilization"]
pa_star = pd.DataFrame(stabilization["by_route"]).T.loc[ROUTE_ORDER]
print(pa_star.assign(tau2=pa_star["tau2"].map("{:.6f}".format),
                     pa_star_weak_side=pa_star["pa_star_weak_side"].round(0),
                     pa_star_both_sides=pa_star["pa_star_both_sides"].round(0)).to_string())
print("\nper-PA noise variance vs LHP:", round(stabilization["per_pa_noise_var_vs_LHP"], 4),
      " both sides:", round(stabilization["per_pa_noise_var_both_sides"], 4))
print("observed median exposure:", stabilization["observed_exposure"]["median_denom_L"],
      "PA vs LHP,", stabilization["observed_exposure"]["median_denom_R"], "PA vs RHP")

             tau2  pa_star_weak_side  pa_star_both_sides
B_prime  0.000393              651.0              1317.0
A        0.000117             2184.0              4419.0
B        0.000590              433.0               877.0

per-PA noise variance vs LHP: 0.2558  both sides: 0.5177
observed median exposure: 86.0 PA vs LHP, 223.0 PA vs RHP


### Route C, bounded and retired

E.15 reported a **negative** split-half reliability for left-handed hitters (−0.366), which
reads as alarming — a reliability cannot be negative. The spec allowed exactly one bounded
diagnostic: audit the implementation, then simulate the estimator's null and see where the
observed value falls.

The audit found no bug. The simulation explains the number: at these half-length exposures
the split-half estimator is so noisy that the observed raw half correlation (−0.155) sits at
the **18th percentile of the τ²=0 null and inside the τ²=B′ null as well**. It discriminates
nothing. The dramatic −0.366 is Spearman-Brown magnifying an ordinary null draw — the
step-up formula 2r/(1+r) has no reliability interpretation on a negative input.

Route C is retired as evidence in either direction. It was never an estimator, and it is not
one now.

In [9]:
route_c = load_json_m("routes_route_c_diagnostic.json")
print("verdict:", route_c["verdict"])
print("\nobserved raw half correlations:", {stand: round(value, 4)
                                            for stand, value in route_c["observed"].items()})
print("\nwhere the observed value falls in each simulated null:")
print(pd.DataFrame({stand: {f"{null}: percentile": round(entry["percentile"], 1)
                            for null, entry in nulls.items()}
                    for stand, nulls in route_c["located"].items()}).to_string())
print("\naudit finding:", route_c["audit"]["finding"])

verdict: uninformative_inside_both_nulls

observed raw half correlations: {'L': -0.1548, 'R': 0.0536, 'pooled': -0.0363}

where the observed value falls in each simulated null:
                             L     R
tau2_zero: percentile     18.2  66.9
tau2_b_prime: percentile   6.4  40.4

audit finding: no bug found. One caveat, not a bug: Spearman-Brown is applied to a NEGATIVE half correlation, and the step-up formula 2r/(1+r) has no reliability interpretation there — it magnifies −0.155 to −0.366. The magnitude of the negative number is therefore an artifact of stepping up an out-of-domain input; the SIGN is the finding, and the raw half correlation −0.155 is the quantity the simulation locates.


## M.1 — the ceiling table with an opponent in it

E.5's ceiling table scored the embedding model alone, so "41% of the ceiling" had nothing to
be compared against. C.2 — the bivariate empirical-Bayes shrinkage baseline, already
committed — is scored here on identical rows.

**§10 fallback rule 1 fired: C.3-full is omitted.** It has no persisted fitted artifact, and
`gbm.predict` retrains the GBM inside the call, which the spec excluded. Worth stating
honestly: that refit is a supported existing code path needing no new feature code, so this
is a scope boundary the spec pre-set, not something C.3 cannot do.

The model's own row is a literal reproduction of E.5's committed within-stand rank
correlation (gate 3c) — if it were not, the comparison beside it would be meaningless.

In [10]:
print("THE MAIN EXHIBIT -- three models x (pooled + three exposure strata),")
print("both claim-1 metrics per cell, each as a fraction of its own bound (higher is better, 1.0 = at the bound).\n")
exhibit = load_m("differential_exhibit.csv")
print(exhibit[["exhibit_column", "model", "n_hitters",
               "rank_corr_within_stand", "rank_fraction_of_ceiling",
               "rank_fraction_ci_low", "rank_fraction_ci_high",
               "pa_weighted_rmse", "rmse_fraction_of_floor",
               "rmse_fraction_ci_low", "rmse_fraction_ci_high"]]
      .round(4).to_string(index=False))
print("\nbuild:", exhibit["build"].unique().tolist())
print("\nDenominators (held fixed across bootstrap replicates):")
print(load_m("differential_exhibit_denominators.csv")[
    ["exhibit_column", "n_hitters", "tau2_b_prime_mix", "mean_sampling_var",
     "rank_ceiling_mc_spearman", "rank_ceiling_mc_standard_error",
     "rank_ceiling_analytic_pearson", "rmse_noise_floor", "reliability"]]
      .round(5).to_string(index=False))
print()
print(load_m("differential_fraction_of_ceiling.csv")
      .drop(columns=["post_selection_descriptive"]).round(4).to_string(index=False))

THE MAIN EXHIBIT -- three models x (pooled + three exposure strata),
both claim-1 metrics per cell, each as a fraction of its own bound (higher is better, 1.0 = at the bound).

exhibit_column          model  n_hitters  rank_corr_within_stand  rank_fraction_of_ceiling  rank_fraction_ci_low  rank_fraction_ci_high  pa_weighted_rmse  rmse_fraction_of_floor  rmse_fraction_ci_low  rmse_fraction_ci_high
        pooled model_v1_model        545                  0.1626                    0.5258                0.2005                 0.8237            0.0635                  1.0057                0.9479                 1.0742
        pooled   eb_bivariate        545                  0.1458                    0.4715                0.1565                 0.7745            0.0642                  0.9952                0.9393                 1.0595
        pooled       gbm_full        545                  0.1413                    0.4568                0.1516                 0.7529            0.0651 

## M.2 — the ceiling inside each exposure stratum

A pooled ceiling hides the thing the thesis is graded on. Low-exposure hitters have more
sampling variance, so their ceiling is lower — and the fraction of it the model reaches is
a different number in each stratum.

τ² is **not** refit per stratum (a per-stratum refit is a new estimator and was not
authorized); the common B′ fit is applied against each stratum's own sampling-variance
profile. Strata are the frozen `claim1_eval` boundaries and are never redefined.

Two achieved columns are reported. The ceiling is a **within-stand** ceiling, so the
comparable achieved figure is the within-stand rank correlation. The raw column includes the
between-stand main effect — which E.5 showed is most of the model's apparent differential
signal — and is shown only so the gap between the two is visible. Comparing the raw column
against this ceiling produces fractions above 1, which is the error, not a finding.

In [11]:
stratum = load_m("differential_route_diagnostics.csv").set_index("stratum").loc[STRATUM_ORDER]
print(stratum[["n_hitters", "median_denom_L", "median_denom_R", "mean_sampling_var",
               "tau2_stratum_mix", "ceiling_b_prime", "ceiling_b_prime_ci_low",
               "ceiling_b_prime_ci_high", "achieved_rank_corr_raw"]]
      .round(4).to_string())
print("\n(the within-stand numerators and their fractions now live in the exhibit above;")
print(" this file keeps only what no exhibit cell carries -- tau2 bookkeeping, the raw-vs-within-stand")
print(" gap, and the route-A fragility apparatus.)")
print("\nRoute A, per stratum (fragility made visible):")
print(stratum[["tau2_route_a", "route_a_degenerate", "ceiling_route_a",
               "route_a_share_degenerate_in_bootstrap"]].round(6).to_string())
print("\nprecision clause:")
print(json.dumps(summary["precision_clause"], indent=2, default=float))

         n_hitters  median_denom_L  median_denom_R  mean_sampling_var  tau2_stratum_mix  ceiling_b_prime  ceiling_b_prime_ci_low  ceiling_b_prime_ci_high  achieved_rank_corr_raw
stratum                                                                                                                                                                          
low            207            43.0           124.0             0.0060            0.0004           0.2531                  0.2371                   0.2690                  0.3825
medium         167            94.0           281.0             0.0037            0.0004           0.3113                  0.2922                   0.3300                  0.4028
high           171           123.0           315.0             0.0032            0.0004           0.3248                  0.3069                   0.3419                  0.2820

(the within-stand numerators and their fractions now live in the exhibit above;
 this file keeps only what no

## M.3 — the level-side ceiling, for comparison

"41% of the platoon ceiling" is only interpretable next to the same figure on the level
side, where the model is known to work. Same construction: observed between-hitter variance
minus expected sampling noise, over observed.

The terms come from the committed F.5 scoring, recomputed on the M.6 intersection.
`noise_floor` there is an **RMSE**, so E[Var noise] is its square; the no-information rung's
`pa_weighted_rmse` is the raw observed spread and its `model_rmse` is already that spread
deconvolved. Two independent cross-checks are reported beside the primary — the C.2 variance
composition and a game-parity split-half on pooled wOBA.

The ceiling is refit-invariant. The achieved fraction is not, and is kept as a separate
field.

In [12]:
level = load_json_m("level_ceiling_level_ceiling.json")
print("primary, on the M.6 intersection:")
print(json.dumps(level["intersection"], indent=2, default=float))
print("\ncross-checks (true-talent sd / implied ceiling):")
print(pd.DataFrame({
    "primary (M.6)": {"true_talent_sd": level["intersection"]["true_talent_sd"],
                      "ceiling_rank_corr": level["intersection"]["ceiling_rank_corr"]},
    "C.2 composition": {"true_talent_sd": level["cross_check_eb_composition"]["true_talent_sd"],
                        "ceiling_rank_corr": float("nan")},
    "split-half (game parity)": {
        "true_talent_sd": float("nan"),
        "ceiling_rank_corr": level["cross_check_split_half"]["ceiling_rank_corr"]},
}).round(4).to_string())

primary, on the M.6 intersection:
{
  "n_hitters": 545,
  "observed_between_hitter_variance": 0.0016816993016209667,
  "E_var_noise": 0.0007556990189828914,
  "true_talent_variance": 0.0009260002826380753,
  "true_talent_sd": 0.03043025275343725,
  "reliability": 0.5506336844794527,
  "ceiling_rank_corr": 0.7420469557106563,
  "degenerate": false,
  "achieved_rank_corr_weighted": 0.46611786061178606,
  "fraction_of_ceiling": 0.6281514357341259,
  "post_selection_descriptive": true
}

cross-checks (true-talent sd / implied ceiling):
                   primary (M.6)  C.2 composition  split-half (game parity)
true_talent_sd            0.0304           0.0267                       NaN
ceiling_rank_corr         0.7420              NaN                    0.6744


### The by-stand asymmetry, reported and not interpreted

E.15's fraction-of-ceiling split by batter stand (L 0.57, R 0.12) is carried forward
**descriptively only**. The statistic that was supposed to support it is the split-half
reliability just retired above, and the L-vs-R difference was never tested.

In [13]:
print(json.dumps(level["by_stand_fractions_descriptive"], indent=2, default=float))

{
  "L": 0.5690628371928935,
  "R": 0.12158155336698968,
  "caveat": "reported descriptively only. The asymmetry's supporting statistic is broken (E.15's LHB split-half reliability is negative) and the L-vs-R difference was never tested."
}


## M.4 — what the intervals actually cover

The five-seed ensemble's intervals under-cover: at a nominal 95%, measured coverage is
~86%. That gap was already known and its fallback already fired — this is a verification
pass on the M.6 population, not another attempt to close it.

**Every interval displayed anywhere in this project carries its measured coverage, not its
nominal level.** E.14 scored 1,149 side-specific groups; the M.6 hitters' own rows are the
relevant slice for everything above.

In [14]:
coverage = load_m("coverage_labels_coverage_labels.csv")
headline = coverage[(coverage["nominal"] == 0.95)
                    & (coverage["interval"] == "seed_plus_target_noise")]
print(headline[["slice", "n_groups", "nominal", "empirical", "ci_low", "ci_high",
                "gap", "mean_half_width"]].round(4).to_string(index=False))
print("\nreproduces the committed E.14 value:", summary["coverage_labels_coverage"]["reproduces_committed"])
print(summary["coverage_labels_coverage"]["label_for_notebook"])

                 slice  n_groups  nominal  empirical  ci_low  ci_high     gap  mean_half_width
   coverage_all_groups      1149     0.95     0.8695  0.8487   0.8877 -0.0805           0.1050
       m6_intersection      1090     0.95     0.8789  0.8582   0.8969 -0.0711           0.1016
   m6_intersection_low       348     0.95     0.8908  0.8537   0.9194 -0.0592           0.1313
m6_intersection_medium       269     0.95     0.8922  0.8495   0.9239 -0.0578           0.1043
  m6_intersection_high       473     0.95     0.8626  0.8286   0.8907 -0.0874           0.0782

reproduces the committed E.14 value: True
every displayed interval carries its MEASURED coverage, not the nominal level — the plan's own fallback, already fired. No further attempt is made to close the gap.


## M.5 — the specialization gradient is a null

The O.2 result belongs in the measurement story, so it is recorded here rather than left in
the Phase O artifacts alone: embedding norm shrinks and projection moves toward zero as
training exposure grows, monotonically across exposure quintiles. That is the signature of
**shrinkage toward the population mean with more data** — the opposite of hitters
specializing into distinct platoon types as they are observed more.

n=2 seeds, descriptive only, never a promotion criterion.

In [15]:
gradient = json.loads((PHASE_O_DIR / "gradient_b.json").read_text())
print("arm:", gradient["arm"], "| n_seeds:", gradient["n_seeds"], "|", gradient["note"])
for seed, block in gradient["seeds"].items():
    print(f"\n{seed}: norm slope per 1000 pitches "
          f"{block['norm']['slope_per_1000_pitches']:+.5f} "
          f"[{block['norm']['ci_low']:+.5f}, {block['norm']['ci_high']:+.5f}]"
          f"  excludes zero: {block['norm']['excludes_zero']}")
    print(pd.DataFrame(block["by_exposure_quintile"])
          [["quintile", "n", "median_train_pitches", "mean_norm", "mean_projection"]]
          .round(4).to_string(index=False))

arm: selection_lr1e3_warm | n_seeds: 2 | n=2, descriptive only, never a promotion criterion

seed_0: norm slope per 1000 pitches -0.01644 [-0.01837, -0.01465]  excludes zero: True
 quintile   n  median_train_pitches  mean_norm  mean_projection
        1 353                  30.0     0.9155          -0.1926
        2 352                 348.0     0.7271          -0.1810
        3 352                1244.0     0.5948          -0.0698
        4 352                3701.0     0.5573          -0.0125
        5 353               10544.0     0.5668           0.0635

seed_1: norm slope per 1000 pitches -0.01236 [-0.01396, -0.01086]  excludes zero: True
 quintile   n  median_train_pitches  mean_norm  mean_projection
        1 353                  30.0     0.7666          -0.1641
        2 352                 348.0     0.6445          -0.1685
        3 352                1244.0     0.5254          -0.0637
        4 352                3701.0     0.4964          -0.0099
        5 353               

## What this notebook establishes

- **The platoon ceiling is low and the route matters.** Under the pre-registered primary
  route B′ the ceiling on the within-stand rank correlation is ≈0.30 and the model reaches
  ≈49% of it. Under the same-season sensitivity route A the ceiling is ≈0.17 and the model
  reaches ≈88% — but a 3% noise-model error moves route A's ceiling to 0.239 in one
  direction and destroys it entirely (τ² < 0) in the other. The honest headline is a
  **bracket, not a point**, and route A is the fragile end of it.
- **Part of the A-vs-B disagreement is population, not estimator.** Restricting the C.2 fit
  to the hitters who reached the 2024 eval population closes about 42% of the B-to-A gap on
  its own.
- **The model has no edge over C.2 on the differential.** 0.146 against 0.146, on identical
  rows. Whatever the embedding is adding on the level side, it is not adding it here.
- **The stratum the thesis is graded on has the lowest ceiling and the highest fraction of
  it reached** (low: ceiling 0.25, 75% reached; high: ceiling 0.32, 34% reached). The model
  looks closest to the ceiling exactly where the ceiling is least worth reaching.
- **The level side is a different regime.** Ceiling ≈0.74, model at ≈65%. The platoon
  problem is not hard because the model is bad at it; it is hard because the answer key is
  mostly noise.
- **Route C is retired**, the negative reliability explained as an artifact rather than a
  finding.
- **Intervals under-cover** (~86% at a nominal 95%) and are labeled accordingly everywhere.

### What is still open

- The B′ conditioning is real: it is a τ² for hitters who survived to 2024, not a
  general-population τ². The bracket does not close it.
- The rank-vs-Pearson step in the ceiling map is approximate at these exposures and biases
  every fraction a few percent high.
- C.3-full is absent from the M.1 comparison by scope, not by capability.
- The by-stand asymmetry has no surviving supporting statistic.